In [ ]:
from functools import partial
import numpy as np
from lucifex.fdm import GridFunctionSeries, NPyConstantSeries
from lucifex.fem import grid_average
from lucifex.viz import (
    plot_colormap, plot_line, save_figure, create_multifigure,
    plot_colormap_multifigure, plot_line_multifigure,
)
from lucifex.utils.array_utils import as_index
from lucifex.io import create_dir_path, find_dir_paths
from lucifex.utils.py_utils import FrozenDict
from lucifex.sim import GridSimulationFromNPZ
from crocodil.dns.system_a import SYSTEM_A_REFERENCE
from crocodil.theory.system_a.late import LateTimeModel

# searching for all simulations that match
PARAMS_NUMERICAL = FrozenDict(
    c_stabilization=None,
    c_limits=True,
)
DIR_ROOT = create_dir_path(
    PARAMS_NUMERICAL, 
    dir_root='./',
    dir_prefix='data', 
    dir_params=PARAMS_NUMERICAL.keys(), 
)
DIR_FIGS = f'{DIR_ROOT}/figures'

T_STOP = 120.0
sim_dir_paths = find_dir_paths(
    DIR_ROOT, 
    include=f't_stop={T_STOP}_*',
    contains=('CHECKPOINT.h5', 'c.npz'),
)
print('Before parameter selection')
for i in sim_dir_paths: print(i)

# choosing a subset of found simulations
PARAMS_PHYSICAL = SYSTEM_A_REFERENCE.replace(Ra=500)

simulations = GridSimulationFromNPZ.dict_from_dir_paths(
    ('Ra', 'Da'), 
    sim_dir_paths,
    ('c', 's'),
    ('f', 'mD', 'mC', 'uRMS'),
    PARAMS_PHYSICAL,
    lazy=True,
    sorting=lambda d: dict(sorted(d.items(), reverse=False)),
)
print('After parameter selection')
for i in simulations.values(): print(i.dir_path)

# setting utilities for plotting
PARAM_REPR = lambda p: int(p) if isinstance(p, float) and float.is_integer(p) else p
save_figure = partial(
    save_figure, 
    dir_path=DIR_FIGS, 
    prefix='early_similarity', 
    pickle=True,
    file_ext=('svg', 'png'),
)

In [ ]:
sim = simulations[(500.0, 1000.0)]